# SiteSight M4: zero-shot baselines on the validation split (GPU)

Runs the same command-line entry points as the laptop, on the **full validation split**, plus the
larger Florence-2 model. **Runtime > Change runtime type > T4 GPU.** On a T4 each model should take minutes
rather than the hours the laptop CPU needs (the laptop's Florence-2 vision encoder alone takes about 45 s per image).

Needs the Colab secret `GITHUB_TOKEN` (read-only, this repo only), as in notebook 01.
The test split is **never** used here: `python -m sitesight.baselines.run` refuses it.

In [ ]:
import base64, os, subprocess
from google.colab import userdata

REPO, DEST = "https://github.com/hddaghigh/sitesight.git", "/content/sitesight"
token = userdata.get("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
header = f"http.extraheader=AUTHORIZATION: basic {auth}"   # keeps the token out of the remote URL
cmd = (["git", "-C", DEST, "-c", header, "pull", "--ff-only"] if os.path.isdir(DEST)
       else ["git", "-c", header, "clone", "--depth", "1", REPO, DEST])
r = subprocess.run(cmd, capture_output=True, text=True)
print("repo ready" if r.returncode == 0 else "git failed: " + (r.stderr or r.stdout).replace(token, "***"))
print(subprocess.run(["git", "-C", DEST, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
%cd /content/sitesight
%pip -q install -e . transformers scipy
import torch
assert torch.cuda.is_available(), "switch the runtime to a GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
DATA = "/content/sitesight_data"
!python scripts/download_data.py --data-dir {DATA} --min-free-gb 3
!python -m sitesight.data.via_parse --via {DATA}/extracted/via_region_data_final.json \
    --images {DATA}/extracted/UAVdata --out {DATA}/derived/annotations.json
!python -m sitesight.data.caption_fields --annotations {DATA}/derived/annotations.json \
    --out {DATA}/derived/annotations_fields.json
# Reuse the laptop's exact split (no embeddings needed, so CPU/GPU float differences cannot change it).
!python -m sitesight.data.splits --annotations {DATA}/derived/annotations_fields.json \
    --splits reports/m2/splits.json --out {DATA}/derived/annotations_curated.json

In [ ]:
ANN, IMG, PRED = f"{DATA}/derived/annotations_curated.json", f"{DATA}/extracted/UAVdata", f"{DATA}/predictions"
!mkdir -p {PRED}
# Full validation split, GPU. Resumable: re-running continues where it stopped.
!python -m sitesight.baselines.run --model owlv2 --annotations {ANN} --images {IMG} --split val \
    --device cuda --save-every 20 --out {PRED}/owlv2_val.json
!python -m sitesight.baselines.run --model florence --annotations {ANN} --images {IMG} --split val \
    --device cuda --save-every 20 --out {PRED}/florence2_base_val.json

In [ ]:
# Size-vs-accuracy: the same two Florence-2 tasks with the 0.77B model (MIT).
!python -m sitesight.baselines.run --model florence --model-id florence-community/Florence-2-large \
    --annotations {ANN} --images {IMG} --split val --device cuda --save-every 20 --out {PRED}/florence2_large_val.json

In [ ]:
!mkdir -p reports/colab
!python -m sitesight.eval.run --annotations {ANN} --split val --common-images \
    --predictions {PRED}/owlv2_val.json {PRED}/florence2_base_val_grounding.json {PRED}/florence2_base_val_regions.json \
                  {PRED}/florence2_large_val_grounding.json {PRED}/florence2_large_val_regions.json \
    --out reports/colab/zero_shot_val_gpu.json

In [ ]:
# Bring back metrics + predictions only (no model weights, no images). OWLv2 stores every candidate box,
# so its JSON is large (hundreds of MB raw); zipped, expect a download of tens of MB.
!cd /content/sitesight && zip -qr /content/m4_outputs.zip reports/colab {PRED}/*.json
from google.colab import files
files.download("/content/m4_outputs.zip")